# 10 . Scenario Lab

A run is four named, **versioned** pieces:

> **Scenario** (entry) × **Stop** × **TP** × **Secondary** (post-entry rules) × **params**

The setup is a **support-break short**: price above a smoothed 200-EMA, HH/HL into resistance with the
lows trapped on a support, then a roll-over below the EMA and finally a **break of that support**
(**level B**) → short.

**Layout** — §1 data · §2 strategy · §3 indicators · §4 rules & execution · §5 chart.
§2–§3 say *what to compute*, §4 says *how to trade it*, §5 is display only.

> Logic lives in `quant/scenarios/`, charting in `quant/viz/studies.py` — this notebook only declares.

## 0 . Setup

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.data import get_ohlcv
from quant import scenarios as sc
from quant.viz import scenario_figure, open_in_browser, CROSSHAIR_JS
pd.set_option('display.max_rows', 120); pd.set_option('display.width', 200)

## 1 . Data

In [2]:
SYMBOL, SOURCE, MARKET, TF = 'BTCUSDT', 'binance', 'spot', '1m'
START, END = '2026-07-25', '2026-07-30'

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} {TF} bars   {df.t.min()}  ->  {df.t.max()}')

[20:06:05] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load


7,201 1m bars   2026-07-25 00:00:00+00:00  ->  2026-07-30 00:00:00+00:00


## 2 . Strategy — the four pieces
Each is `name@version`. `sc.SCENARIOS` / `STOP_METHODS` / `TP_METHODS` / `SECONDARIES` list them all.

In [3]:
SCENARIO  = 'support_break_short@v1.2'  # v1.0 core | v1.1 +structure | v1.2 no-structure | v1.3 nearest-trough
STOP      = 'leg_high@v1'               # pct | swing_high | prev_high | leg_high | high_of_n | zone_top | atr
TP        = 'rr@v1'                     # none | rr | fixed_pct | trailing | opposite_candle |
                                        #   n_green_exit | back_above_ema | next_zone | rr_partials
SECONDARY = [                           # post-entry rules; a LIST runs them together. None = off.
    'shaky_trend_bail@v1',              #   back at the EMA -> bail on next opposite candle, if green
    'run_past_tp@v1',                   #   reaching TP ARMS instead of closing -> ride the move on
]

for k, res in [(SCENARIO, sc.resolve_scenario), (STOP, sc.resolve_stop), (TP, sc.resolve_tp)]:
    print(f'{k:30s} {res(k).description}')

support_break_short@v1.2       No-structure: no S/R zones — N consecutive red candles below the EMA and a break of the lowest low of the prior N bars.
leg_high@v1                    Stop above the high of the CURRENT down-leg — the running max since price last closed below the EMA line. No confirmation lag, so it tracks the high a human would draw; set stop_above_ema_only to use only highs that poked above the EMA.
rr@v1                          Fixed reward:risk target (R multiple of the stop distance).


## 3 . Indicators — what gets computed
Only the *definitions* of lines and levels. Nothing here decides when to trade (that is §4).

In [4]:
INDICATORS = dict(
    # the main EMA + its slope (red dotted line on price, and the gradient pane)
    grad_period=200, grad_smooth=9,     # the "200 EMA smoothed by 9" line
    grad_lookback=5,                    # bars used to measure its slope
    # S/R zones (used by scenario v1.0 / v1.1 / v1.3)
    sr_left=10, sr_right=10, sr_cluster_pct=0.15,
)

# Higher-timeframe EMA LINES drawn on this chart (lines only — no extra pane).
# build_mtf shifts each HTF feature one HTF bar before merging, so a value shown on a 1m bar was
# already CLOSED at that moment — no leak from the unfinished 5m candle.
#   ('5min', 200)  |  ('5min', [100, 200])  |  [('5min',100), ('15min',200)]  |  None = off
MTF_EMAS = ('5min', [100, 200])

ATR_PERIOD = 14        # ATR stop, ATR bands, and the zigzag structure threshold
RSI_PERIOD = 14

# Market structure (HH / HL / LH / LL)
STRUCT_METHOD   = 'zigzag'   # 'zigzag' = ATR-adaptive, ~2-4 bar lag | 'fractal' = fixed STRUCT_LR lag
STRUCT_ATR_MULT = 2.5        # zigzag threshold; bigger = fewer, more significant swings
STRUCT_LR       = 10         # fractal only: look-around (== its lag in bars)

# RSI divergence pivot. Lag is exactly DIV_RIGHT bars; DIV_LEFT is FREE (already history).
DIV_LEFT, DIV_RIGHT = 10, 2
RSI_DIV_MODE = 'confirmed'   # 'confirmed' (lag=DIV_RIGHT) | 'realtime' (zero-lag, ~53% ratified) | 'both'

## 4 . Rules & execution

**4a** adds your own entry conditions; **4b** sets every strategy parameter and runs the backtest.


### 4a . Extra entry rules — your own conditions

Anything here is **ANDed onto** the scenario's own trigger, so it can only ever *remove* trades.
Conditions combine with plain Python operators — `&` = AND, `|` = OR, `~` = NOT, `()` to group:

```python
ENTRY_RULES = (below_mtf('5min', 200, n=3) & red(2)) | below_mtf('15min', 200, n=5)
```

**Building blocks** (all take `src='ha'` for Heikin-Ashi):

| condition | meaning |
|---|---|
| `below_mtf(tf, period, n, mode)` | price below a higher-TF EMA for `n` bars. `mode='close'` (closes) or `'full'` (whole candle) |
| `above_mtf(tf, period, n, mode)` | the mirror image |
| `red(n)` / `green(n)` | `n` consecutive red / green candles |
| `below(a, b)` / `above(a, b)` | a column or number vs another |
| `full_below(level, n)` / `full_above(level, n)` | whole candle beyond a level for `n` bars |
| `near_support(pct)` / `near_resistance(pct)` | price into an S/R zone |

> The timeframes you reference must be listed in `MTF_EMAS` (§3) so the columns exist.
> `ENTRY_RULES` is code, not data — it is **not** saved into a Preset; pass it at run time.

In [5]:
from quant.strategies.conditions import (below_mtf, above_mtf, red, green,
                                         below, above, full_below, full_above)

# None = no extra rules. Edit freely — grouping with () works exactly as you'd expect.
ENTRY_RULES = None# below_mtf('5min', 200, n=1) & red(1)

# other shapes to try:
#   ENTRY_RULES = below_mtf('5min', 200, n=3, mode='full')          # whole candles below the 5m EMA
#   ENTRY_RULES = (below_mtf('5min',200,n=3) & red(2)) | below_mtf('5min',100,n=5)
#   ENTRY_RULES = below_mtf('5min',200) & ~green(1)                 # NOT a green candle

#print('extra entry rule:', ENTRY_RULES.name if ENTRY_RULES is not None else '(none)')

In [6]:
# --- 4b -------------------------------------------------------------------------------------
PARAMS = dict(
    **INDICATORS,
    mtf_emas=MTF_EMAS,          # build the higher-TF EMA columns so ENTRY_RULES can use them


    # -- ENTRY: what counts as a trigger --------------------------------------------------------
    entry_mode='break_close',   # break_close | wait_n | n_red | pullback
    break_mode='close_below',   # candle CLOSES below level B | 'full_below' = whole candle below it
    prev_low_n=20,              # v1.2: level B = lowest low of the prior N bars
    sr_min_touches=1,           # v1.0/v1.1: min touches for a support zone to count
    grad_max=0.0,               # gradient THRESHOLD: entry needs grad < this. None = no gate.
    break_min_ema_dist_pct=None,  # reject level B if TOO CLOSE to the EMA line (%)
    break_max_ema_dist_pct=None,  # reject level B if TOO FAR from the EMA line (%)

    # -- CANDLES: source + the N counts ----------------------------------------------------------
    candle_src='real',          # 'real' | 'ha'  -> ALL candle colour/shape tests use this source
    confirm_red=2,              # v1.2 context: N red candles below the EMA
    n_red=1,                    # entry_mode='n_red'   : N red candles after the break
    wait_n=3,                   # entry_mode='wait_n'  : N bars to wait after the break
    pullback_lookback=5,        # entry_mode='pullback': N bar window for the retest

    # -- STOP -------------------------------------------------------------------------------------
    stop_buffer_pct=0.05,       # buffer beyond the structural level (%)
    stop_max_risk_pct=0.0,      # cap: structural stop risking more than this % -> use stop_pct
    stop_pct=0.6,               # the 'pct' stop, and the fallback for structural stops
    stop_above_ema_only=False,  # leg_high: count only highs that poked ABOVE the EMA line
    stop_lr=3,                  # prev_high: pivot look-around (costs `lr` bars of lag)
    stop_swing_lr=12,           # swing_high: pivot look-around
    stop_n=20,                  # high_of_n: N bars
    stop_atr_period=ATR_PERIOD, stop_atr_mult=1.5,

    # -- TAKE PROFIT --------------------------------------------------------------------------------
    tp_rr=2.0,                  # rr / rr_partials: R multiple
    tp_pct=0.6,                 # fixed_pct
    trail_pct=1.0,              # trailing
    tp_opp_n=1,                 # opposite_candle: close after N opposite-colour candles
    tp_zone_pct=0.1,            # next_zone

    # -- SECONDARY rules -----------------------------------------------------------------------------
    sec_arm_mode='near',        # how close to the EMA arms a trade:
    sec_arm_tol_pct=0.05,       #   'near' (within this % BELOW) | 'touch' | 'close_above' | 'full_above'
    sec_opp_n=1,                # exit after N opposite-colour candles
    sec_replace_tp=False,       # once armed, also drop the main TP (False = keep both, first wins)
    sec_arm_on_tp=True,         # run_past_tp: reaching TP arms instead of closing

    # -- WHEN to trade. Block-lists exclude, allow-lists keep. None = off. ----------------------------
    block_times=('08:00', '13:00'),  # ('HH:MM','HH:MM'), or a tuple of them -> no entries then
    block_hours=None,           # e.g. (12, 16, 18)  -> no entries in those clock hours
    block_weekdays=None,        # e.g. (5, 6)        -> no entries Sat/Sun (0=Mon .. 6=Sun)
    hours=None,                 # ALLOW-list: (7, 16) keeps ONLY 07:00-16:00
    weekdays=None,              # ALLOW-list: (0,1,2,3,4) keeps ONLY Mon-Fri
    session=None,               # 'london' | 'newyork' | 'tokyo' | 'sydney'
    filter_tz='UTC',            # evaluate all of the above in this timezone
)

CFG = dict(initial_cash=10_000, spread=0.0,
           sizing_mode='risk_pct_equity', sizing_value=1.0)
# To actually risk sizing_value% you need BOTH of these — max_notional_pct defaults to 100, which
# silently caps a tight-stop position at ~1x equity:   allow_leverage=True, max_notional_pct=5000

run = sc.run_pieces(df, SCENARIO, STOP, TP, secondary=SECONDARY,
                    entry_filter=ENTRY_RULES, params=PARAMS, cfg=CFG)
s = run.stats
print(f"trades {int(s['num_trades'])} | win {s['win_rate_pct']:.0f}% | ret {s['total_return_pct']:+.2f}% "
      f"| dd {s['max_drawdown_pct']:.2f}% | PF {s['profit_factor']:.2f} | Sharpe {s['sharpe']:.2f} "
      f"| kernel {run.result.elapsed_s*1e3:.0f} ms")
print('close reasons:', dict(run.trades.close_reason.value_counts()) if len(run.trades) else {})
run.trades[['side','entry_time','exit_time','entry_price','stop_price','exit_price',
            'pnl','return_pct','bars_held','close_reason']]

[20:06:06] INFO | quant.engine | backtest done | bars=7,201 trades=19 final_cash=10211.06 return=2.11% dd=0.55% | 32.8 ms


trades 19 | win 84% | ret +2.11% | dd 0.55% | PF 4.32 | Sharpe 12.14 | kernel 33 ms
close reasons: {'alt_exit': np.int64(16), 'stop_loss': np.int64(3)}


,side,entry_time,exit_time,entry_price,stop_price,exit_price,pnl,return_pct,bars_held,close_reason
0,short,2026-07-25 05:38:00+00:00,2026-07-25 06:17:00+00:00,64035.90,64165.576755,64033.990000,0.298270,0.002983,39,alt_exit
1,short,2026-07-25 16:25:00+00:00,2026-07-25 16:47:00+00:00,64123.01,64218.093000,64218.093000,-14.828661,-0.148282,22,stop_loss
2,short,2026-07-25 20:38:00+00:00,2026-07-25 21:07:00+00:00,64323.01,64403.535675,64319.320000,0.572834,0.005737,29,alt_exit
3,short,2026-07-26 00:01:00+00:00,2026-07-26 00:07:00+00:00,64356.01,64418.363085,64418.363085,-9.675251,-0.096888,6,stop_loss
4,short,2026-07-26 05:27:00+00:00,2026-07-26 05:34:00+00:00,64500.00,64552.260000,64467.400000,5.042319,0.050543,7,alt_exit
5,short,2026-07-26 07:10:00+00:00,2026-07-26 07:22:00+00:00,64398.88,64478.243010,64376.200000,3.515253,0.035218,12,alt_exit
6,short,2026-07-26 17:40:00+00:00,2026-07-26 20:20:00+00:00,64668.91,64780.374000,64650.610000,2.825533,0.028298,160,alt_exit
7,short,2026-07-27 00:54:00+00:00,2026-07-27 01:14:00+00:00,65117.99,65228.427915,65057.840000,9.225764,0.092371,20,alt_exit
8,short,2026-07-27 07:37:00+00:00,2026-07-27 09:25:00+00:00,65282.61,65459.383335,65225.220000,8.788350,0.087910,108,alt_exit
9,short,2026-07-27 14:29:00+00:00,2026-07-27 16:28:00+00:00,64870.69,65288.617995,64753.990000,18.000004,0.179896,119,alt_exit


## 5 . Chart
Display toggles only — none of these change a trade. Every layer is also its own legend entry, so you
can switch things on and off in the browser without re-running.

In [7]:
SHOW_MTF_EMAS  = True    # the higher-TF EMA lines from §3
SHOW_ATR_BANDS = None    # (1,2,3,4) -> EMA +/- k*ATR envelopes; None = off
SHOW_STRUCTURE = False   # HH/HL/LH/LL labels + trend ribbon
SHOW_RSI       = False   # adds an RSI pane
SHOW_RSI_DIV   = False   # divergences (needs SHOW_RSI)

fig = scenario_figure(
    run,
    mtf_ema=(MTF_EMAS if SHOW_MTF_EMAS else None), mtf_ema_pane=False,
    atr_bands=SHOW_ATR_BANDS, atr_period=ATR_PERIOD,
    structure=SHOW_STRUCTURE, struct_method=STRUCT_METHOD,
    struct_atr_mult=STRUCT_ATR_MULT, struct_lr=STRUCT_LR,
    rsi=SHOW_RSI, rsi_period=RSI_PERIOD,
    rsi_divergence=(RSI_DIV_MODE if SHOW_RSI_DIV else False),
    div_left=DIV_LEFT, div_right=DIV_RIGHT,
    height=1150 if SHOW_RSI else 900,
)
print('chart ->', open_in_browser(fig, 'scenario_chart.html', extra_js=CROSSHAIR_JS))

chart -> C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\scenario_chart.html


**On the chart:** candles · red-dotted EMA line · **level B** (amber dashed — the level each trade
actually broke) · **entry** ▼ · **stop** (red dotted) · **exit** coloured by reason · purple ▲ where a
secondary armed · the gradient pane · plus whatever you switched on above.

**Mind the lag.** Anything pivot-based — `swing_high` / `prev_high` stops, `fractal` structure,
confirmed divergence — can never be fresher than its right-hand confirmation. `leg_high`, `high_of_n`
and the `zigzag` structure avoid or minimise it.